# 📖 Notebook 5: Capacity, TTL, Transactions & Hot Partitions

So far we've covered **modeling** (keys, indexes, single-table design) and **CDC** (streams). This notebook covers the everyday features that keep a DynamoDB-backed system **correct, fast, and cheap in production**.

Each section follows a **bad practice → best practice** progression, so you can *feel* the problem before seeing the fix.

## Learning Objectives

By the end of this notebook, you'll understand:
- On-demand vs provisioned **capacity modes** and when to choose each
- How **TTL** auto-expires items for free (no scan-and-delete jobs)
- How **conditional writes** prevent the "lost update" race condition
- How **transactions** (`TransactWriteItems`) give you ACID across multiple items
- Why **hot partitions** throttle your workload — and how **write sharding** fixes it
- What **DAX** is, when to reach for it, and the cache-aside pattern


## 🛠️ Setup

Start the infrastructure first:

```bash
cd 03-technologies/databases/dynamodb
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

> ⚠️ **Note on DynamoDB Local**: Some production-only features (TTL auto-expiry, DAX, true throttling under hot partitions) don't execute locally. We **simulate** them so you can see the API surface and the *pattern* — the concepts transfer 1:1 to real AWS.


In [ ]:
import boto3
from boto3.dynamodb.conditions import Key, Attr
from botocore.exceptions import ClientError
import json
import time
import random
from decimal import Decimal

dynamodb = boto3.resource(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

client = boto3.client(
    "dynamodb",
    endpoint_url="http://localhost:8000",
    region_name="us-east-1",
    aws_access_key_id="local",
    aws_secret_access_key="local",
)

try:
    client.list_tables()
    print("✅ Connected to DynamoDB Local")
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")


## 💰 Part 1: Capacity Modes — On-Demand vs Provisioned

Every DynamoDB table has a **billing mode** that decides how you pay for reads and writes.

| Mode | How You Pay | Best For |
|------|-------------|----------|
| **On-demand** (`PAY_PER_REQUEST`) | Per request. DynamoDB auto-scales instantly. | Unpredictable traffic, new apps, spiky workloads |
| **Provisioned** (`PROVISIONED`) | You pre-purchase RCU/WCU per second. Cheaper at steady high load. | Predictable workloads, cost-sensitive at scale |

**Capacity units:**
- **1 RCU** = 1 strongly-consistent read/sec of a 4KB item (or 2 eventually-consistent reads)
- **1 WCU** = 1 write/sec of a 1KB item

### ❌ Bad: Provisioned with tiny capacity for a spiky app

Imagine a launch day: 50 RCU provisioned, then 10× traffic spike → throttling, 5xx errors for users.

### ✅ Best: Pick based on your traffic shape

- **New / unpredictable / bursty** → start with **on-demand**. You pay a premium per request but never get throttled, and you can switch to provisioned later once traffic stabilizes.
- **Steady, predictable** (e.g., a backend that writes 500 WCU at all hours) → **provisioned with auto-scaling**. Typically 5-7× cheaper than on-demand at high utilization.

> 💡 You can switch between modes **once every 24 hours**. So pick one, measure, then adjust — don't agonize on day 1.


In [ ]:
# Compare the two modes side by side (DynamoDB Local accepts both but bills nothing)

# ❌ BAD shape: tiny provisioned capacity on a table that will see spikes
try:
    dynamodb.Table("ProvisionedTooSmall").delete()
    dynamodb.Table("ProvisionedTooSmall").wait_until_not_exists()
except ClientError:
    pass

dynamodb.create_table(
    TableName="ProvisionedTooSmall",
    KeySchema=[{"AttributeName": "id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "id", "AttributeType": "S"}],
    BillingMode="PROVISIONED",
    ProvisionedThroughput={"ReadCapacityUnits": 5, "WriteCapacityUnits": 5},  # tiny!
).wait_until_exists()
print("⚠️  Created 'ProvisionedTooSmall' with 5 RCU / 5 WCU")
print("   → Any spike above 5 writes/sec will throttle users.")

# ✅ BEST shape: on-demand, no capacity planning
try:
    dynamodb.Table("OnDemandSafe").delete()
    dynamodb.Table("OnDemandSafe").wait_until_not_exists()
except ClientError:
    pass

dynamodb.create_table(
    TableName="OnDemandSafe",
    KeySchema=[{"AttributeName": "id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
).wait_until_exists()
print("✅ Created 'OnDemandSafe' with on-demand billing")
print("   → Scales instantly, you pay per request.")

print()
print("💡 Rule of thumb:")
print("   • Unknown traffic?   → on-demand (safer, easy)")
print("   • Known, steady?     → provisioned + auto-scaling (cheaper)")


## ⏰ Part 2: TTL — Auto-Expire Items for Free

Many items only matter for a limited time: **web sessions**, **password reset tokens**, **OTP codes**, **temporary caches**, **rate-limit counters**. You don't want them sitting in the table forever.

### ❌ Bad: A nightly "cleanup" cron that scans and deletes expired items

Problems:
- Scan reads **every** item in the table — expensive.
- Burns WCU on every delete.
- Late cleanup means expired tokens stay valid for hours.
- One more service to build, monitor, and page you at 3am.

### ✅ Best: DynamoDB TTL

1. Pick a numeric attribute (e.g., `expires_at`) containing a **Unix epoch timestamp in seconds**.
2. Enable TTL on the table pointing at that attribute.
3. DynamoDB deletes items on its own within ~48h of the expiry time, **for free** (no RCU/WCU charge for the delete).

> ⚠️ TTL is "eventually deleted". Don't rely on it for security-critical expiry — double-check the timestamp at read time too.


In [ ]:
# Set up a Sessions table and configure TTL on the 'expires_at' attribute

TABLE_NAME = "UserSessions"

try:
    dynamodb.Table(TABLE_NAME).delete()
    dynamodb.Table(TABLE_NAME).wait_until_not_exists()
except ClientError:
    pass

sessions = dynamodb.create_table(
    TableName=TABLE_NAME,
    KeySchema=[{"AttributeName": "session_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "session_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
sessions.wait_until_exists()

# Enable TTL (in real AWS this takes a few minutes to activate)
try:
    client.update_time_to_live(
        TableName=TABLE_NAME,
        TimeToLiveSpecification={"Enabled": True, "AttributeName": "expires_at"},
    )
    print(f"✅ Enabled TTL on '{TABLE_NAME}' using attribute 'expires_at'")
except ClientError as e:
    # DynamoDB Local sometimes rejects repeated updates — safe to ignore
    print(f"ℹ️  TTL update: {e.response['Error']['Code']}")

# Insert a session that expires in 1 hour
now = int(time.time())
sessions.put_item(Item={
    "session_id": "sess-alice-123",
    "user_id": "user-001",
    "expires_at": now + 3600,  # Unix seconds, 1 hour from now
})
# Insert an already-expired session (simulates what TTL would eventually clean up)
sessions.put_item(Item={
    "session_id": "sess-expired-999",
    "user_id": "user-002",
    "expires_at": now - 60,
})

print()
print("📋 Items stored:")
for item in sessions.scan()["Items"]:
    ttl_in = int(item["expires_at"]) - now
    state = "expired" if ttl_in < 0 else f"expires in {ttl_in}s"
    print(f"   {item['session_id']:<20} → {state}")
print()
print("💡 In real AWS, the expired session disappears on its own within ~48h.")
print("   No cleanup job, no Scan, no WCU cost.")


In [ ]:
# Defensive read: always check the timestamp at read time too.
# TTL is eventual — assume an expired item could still be returned briefly.

def get_valid_session(session_id):
    resp = sessions.get_item(Key={"session_id": session_id})
    item = resp.get("Item")
    if not item:
        return None
    if int(item["expires_at"]) < int(time.time()):
        return None  # Treat as expired even if TTL hasn't cleaned it up yet
    return item

print("Valid session:   ", get_valid_session("sess-alice-123"))
print("Expired session: ", get_valid_session("sess-expired-999"))
print()
print("💡 Belt and suspenders: TTL saves storage, the read-time check protects correctness.")


## 🔒 Part 3: Conditional Writes — Preventing Lost Updates

Two tabs open on the same shopping cart. Both read `quantity=1`, both write `quantity+=1`, both save. Final value: `2` — but the user clicked `+` **twice**, so they expected `3`. This is the classic **lost update** race.

### ❌ Bad: Read → modify → write (no condition)


In [ ]:
# ❌ BAD: read-modify-write with no concurrency control.
# We simulate two concurrent clients both incrementing quantity.

try:
    dynamodb.Table("Cart").delete()
    dynamodb.Table("Cart").wait_until_not_exists()
except ClientError:
    pass

cart = dynamodb.create_table(
    TableName="Cart",
    KeySchema=[{"AttributeName": "cart_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "cart_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
cart.wait_until_exists()

cart.put_item(Item={"cart_id": "cart-1", "quantity": 1, "version": 1})

def naive_increment(cart_id):
    current = cart.get_item(Key={"cart_id": cart_id})["Item"]
    new_qty = current["quantity"] + 1
    # ⏳ Imagine another client sneaking in here and writing first...
    cart.put_item(Item={"cart_id": cart_id, "quantity": new_qty, "version": 1})

# Both clients read qty=1, both write qty=2 → second +1 is LOST
naive_increment("cart-1")
naive_increment("cart-1")

print("After TWO naïve increments:", cart.get_item(Key={"cart_id": "cart-1"})["Item"])
print("😱 Expected quantity=3, got quantity=2. The second update was lost.")


### ✅ Best #1: Atomic counter with `ADD` / `SET x = x + :n`

For simple numeric increments, let DynamoDB do the math server-side. No read needed, no race.


In [ ]:
# Reset
cart.put_item(Item={"cart_id": "cart-1", "quantity": 1, "version": 1})

def atomic_increment(cart_id, delta=1):
    cart.update_item(
        Key={"cart_id": cart_id},
        UpdateExpression="SET quantity = quantity + :d",
        ExpressionAttributeValues={":d": delta},
    )

atomic_increment("cart-1")
atomic_increment("cart-1")

print("After TWO atomic increments:", cart.get_item(Key={"cart_id": "cart-1"})["Item"])
print("✅ quantity=3 as expected — DynamoDB applied both increments atomically.")


### ✅ Best #2: Optimistic locking with a `version` attribute

For **non-counter updates** (e.g., "set status=SHIPPED only if currently PENDING"), use a `ConditionExpression`.
If the condition fails, you get a `ConditionalCheckFailedException` and can retry or show "please refresh".

This is the same idea as SQL's `UPDATE ... WHERE version = ?` optimistic concurrency.


In [ ]:
# Reset
cart.put_item(Item={"cart_id": "cart-1", "quantity": 1, "version": 1})

def safe_update(cart_id, new_qty, expected_version):
    try:
        cart.update_item(
            Key={"cart_id": cart_id},
            UpdateExpression="SET quantity = :q, version = version + :one",
            ConditionExpression="version = :v",   # only proceed if version matches
            ExpressionAttributeValues={
                ":q": new_qty,
                ":v": expected_version,
                ":one": 1,
            },
        )
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "ConditionalCheckFailedException":
            return False
        raise

# Client A reads version=1, Client B reads version=1.
# Client A commits first → version becomes 2.
print("Client A writes qty=10:", safe_update("cart-1", 10, expected_version=1))
# Client B tries to commit with its stale version=1 → rejected.
print("Client B writes qty=99:", safe_update("cart-1", 99, expected_version=1))

print()
print("Final state:", cart.get_item(Key={"cart_id": "cart-1"})["Item"])
print()
print("💡 Client B should re-read the item and retry with the new version.")
print("   This is how you prevent lost updates on non-counter fields.")


### ✅ Best #3: Idempotent writes with `attribute_not_exists`

A common bug: the user double-clicks "Place Order" and you create two orders.
Fix with a condition that the primary key must not already exist:

```python
table.put_item(
    Item={"order_id": "ORD-123", ...},
    ConditionExpression="attribute_not_exists(order_id)",
)
```

Now the second click fails cleanly with `ConditionalCheckFailedException` and you can tell the user "Order already placed."


## 🔀 Part 4: Transactions — ACID Across Multiple Items

A bank transfer touches **two** accounts: debit Alice, credit Bob. If the second write fails, Alice loses money that Bob never received. You need **atomicity** across both items.

### ❌ Bad: Two separate `update_item` calls


In [ ]:
# ❌ BAD: non-atomic transfer. If the second call fails, money vanishes.

try:
    dynamodb.Table("Accounts").delete()
    dynamodb.Table("Accounts").wait_until_not_exists()
except ClientError:
    pass

accounts = dynamodb.create_table(
    TableName="Accounts",
    KeySchema=[{"AttributeName": "account_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "account_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
accounts.wait_until_exists()

accounts.put_item(Item={"account_id": "alice", "balance": Decimal("100")})
accounts.put_item(Item={"account_id": "bob",   "balance": Decimal("50")})

def bad_transfer(src, dst, amount):
    accounts.update_item(
        Key={"account_id": src},
        UpdateExpression="SET balance = balance - :a",
        ExpressionAttributeValues={":a": Decimal(str(amount))},
    )
    # 💥 If the process crashes here, the money is gone.
    accounts.update_item(
        Key={"account_id": dst},
        UpdateExpression="SET balance = balance + :a",
        ExpressionAttributeValues={":a": Decimal(str(amount))},
    )

bad_transfer("alice", "bob", 30)
print("After bad transfer:")
print(" ", accounts.get_item(Key={"account_id": "alice"})["Item"])
print(" ", accounts.get_item(Key={"account_id": "bob"})["Item"])
print("⚠️  Looks fine here — but in production, a network failure between the two calls loses money.")


### ✅ Best: `TransactWriteItems` — all-or-nothing across up to 100 items

DynamoDB transactions give you ACID semantics. Either **every** write succeeds, or **none** do. You can combine `Put`, `Update`, `Delete`, and `ConditionCheck` operations across **different items and even different tables**.

Limits: up to 100 items, 4 MB total, costs 2× the normal WCU (double writes behind the scenes for the commit protocol).


In [ ]:
# ✅ BEST: atomic transfer using TransactWriteItems + preconditions.

# Reset balances
accounts.put_item(Item={"account_id": "alice", "balance": Decimal("100")})
accounts.put_item(Item={"account_id": "bob",   "balance": Decimal("50")})

def transfer(src, dst, amount):
    try:
        client.transact_write_items(
            TransactItems=[
                {
                    "Update": {
                        "TableName": "Accounts",
                        "Key": {"account_id": {"S": src}},
                        "UpdateExpression": "SET balance = balance - :a",
                        # Don't allow overdraft!
                        "ConditionExpression": "balance >= :a",
                        "ExpressionAttributeValues": {":a": {"N": str(amount)}},
                    }
                },
                {
                    "Update": {
                        "TableName": "Accounts",
                        "Key": {"account_id": {"S": dst}},
                        "UpdateExpression": "SET balance = balance + :a",
                        "ExpressionAttributeValues": {":a": {"N": str(amount)}},
                    }
                },
            ]
        )
        return True
    except ClientError as e:
        # TransactionCanceledException is what you'll see when a condition fails
        print(f"   ❌ Transfer cancelled: {e.response['Error']['Code']}")
        return False

print("Transfer $30 alice → bob:", transfer("alice", "bob", 30))
print(" ", accounts.get_item(Key={"account_id": "alice"})["Item"])
print(" ", accounts.get_item(Key={"account_id": "bob"})["Item"])

print()
print("Transfer $9999 alice → bob (overdraft!):", transfer("alice", "bob", 9999))
print(" ", accounts.get_item(Key={"account_id": "alice"})["Item"], "← untouched")
print(" ", accounts.get_item(Key={"account_id": "bob"})["Item"],   "← untouched")

print()
print("💡 Either both updates happen, or neither. No partial state, ever.")
print("   Great for: payments, inventory reservations, multi-entity workflows.")


## 🔥 Part 5: Hot Partitions & Write Sharding

Every physical partition has a hard ceiling: **~3,000 RCU / 1,000 WCU**. Send more than that to a single partition key and AWS returns `ProvisionedThroughputExceededException` — even if the table's total capacity is huge.

### ❌ Bad: Using a low-cardinality attribute as the partition key

Suppose we track "trending" posts with partition key = the literal string `"trending"`. All writes go to **one** partition → hot partition → throttling.


In [ ]:
# ❌ BAD: every write goes to the same partition key "trending".
# In production this would throttle under load.

try:
    dynamodb.Table("TrendingBad").delete()
    dynamodb.Table("TrendingBad").wait_until_not_exists()
except ClientError:
    pass

bad = dynamodb.create_table(
    TableName="TrendingBad",
    KeySchema=[
        {"AttributeName": "bucket", "KeyType": "HASH"},
        {"AttributeName": "post_id", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "bucket", "AttributeType": "S"},
        {"AttributeName": "post_id", "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",
)
bad.wait_until_exists()

with bad.batch_writer() as batch:
    for i in range(200):
        batch.put_item(Item={"bucket": "trending", "post_id": f"post-{i:04d}", "score": i})

# Show what happened: 200 items all crammed into ONE partition key
partitions = {}
for item in bad.scan()["Items"]:
    partitions.setdefault(item["bucket"], 0)
    partitions[item["bucket"]] += 1

print("📊 Item distribution on the BAD table:")
for k, v in partitions.items():
    print(f"   '{k}' → {v} items on the same partition")
print("⚠️  In real AWS: this is a hot partition. Every write competes for the same 1,000 WCU.")


### ✅ Best: Write Sharding

Append a random suffix (a "shard") to the partition key so writes spread across N partitions. When reading, fan out to all N shards in parallel and merge the results.

```
"trending"           →  "trending#0", "trending#1", ..., "trending#9"
1,000 WCU ceiling    →  10,000 WCU effective (10 partitions × 1,000)
```

Trade-off: reads now need **N Query calls** instead of 1. Size N to your workload (usually 10-100).


In [ ]:
# ✅ BEST: spread writes across 10 shards.

NUM_SHARDS = 10

try:
    dynamodb.Table("TrendingGood").delete()
    dynamodb.Table("TrendingGood").wait_until_not_exists()
except ClientError:
    pass

good = dynamodb.create_table(
    TableName="TrendingGood",
    KeySchema=[
        {"AttributeName": "bucket", "KeyType": "HASH"},
        {"AttributeName": "post_id", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "bucket", "AttributeType": "S"},
        {"AttributeName": "post_id", "AttributeType": "S"},
    ],
    BillingMode="PAY_PER_REQUEST",
)
good.wait_until_exists()

def shard_key(logical_key, post_id):
    # Deterministic sharding on post_id keeps the same post on the same shard (makes updates easy).
    # Random sharding spreads hottest writes even better.
    suffix = hash(post_id) % NUM_SHARDS
    return f"{logical_key}#{suffix}"

with good.batch_writer() as batch:
    for i in range(200):
        post_id = f"post-{i:04d}"
        batch.put_item(Item={
            "bucket": shard_key("trending", post_id),
            "post_id": post_id,
            "score": i,
        })

partitions = {}
for item in good.scan()["Items"]:
    partitions.setdefault(item["bucket"], 0)
    partitions[item["bucket"]] += 1

print("📊 Item distribution on the GOOD (sharded) table:")
for k in sorted(partitions):
    bar = "█" * partitions[k]
    print(f"   {k:<14} {bar} ({partitions[k]})")
print()
print("✅ Writes spread across 10 partitions → 10× the effective write throughput.")


In [ ]:
# Reading from a sharded bucket: fan out across all shards, then merge.

def read_all_trending():
    all_items = []
    for s in range(NUM_SHARDS):
        resp = good.query(
            KeyConditionExpression=Key("bucket").eq(f"trending#{s}")
        )
        all_items.extend(resp["Items"])
    # Merge and re-sort (e.g., by score desc for a trending feed)
    return sorted(all_items, key=lambda x: -int(x["score"]))

top5 = read_all_trending()[:5]
print("🔥 Top 5 trending posts (merged from 10 shards):")
for item in top5:
    print(f"   {item['post_id']} → score={item['score']}")

print()
print("💡 Fan-out reads cost N Query calls instead of 1. This is the classic")
print("   write-availability vs read-cost trade-off of sharding. In a real system you'd")
print("   run the N queries concurrently (threads / asyncio) to keep latency low.")


## ⚡ Part 6: DAX — The In-Memory Cache for DynamoDB

**DAX (DynamoDB Accelerator)** is a fully-managed, write-through, in-memory cache that sits in front of DynamoDB. Reads hit DAX first; on a miss, DAX calls DynamoDB and caches the result.

| Metric | DynamoDB | DAX |
|---|---|---|
| Read latency | single-digit **milliseconds** | **microseconds** (100-400 µs) |
| Cost model | RCU per read | Instance-hour |
| Consistency | Strong or eventual | Eventual (for `GetItem`/`Query`/`Scan`) |

> ⚠️ DAX is an AWS-managed service. It doesn't run in DynamoDB Local, so we'll walk through the **pattern** conceptually with a local dict cache.

### ❌ Bad: Every request hits DynamoDB, even hot keys

A product detail page fetches the same top-100 products millions of times an hour. Each read burns RCU and adds ~5 ms of network latency.

### ✅ Best: Put DAX in front (or use a cache-aside pattern with Redis/Memcached)

- DAX is an in-VPC cluster — **no code changes** beyond pointing the boto3 client at DAX.
- Write-through: every write goes to DAX *and* DynamoDB, so the cache doesn't serve stale data after updates you control.
- Ideal for read-heavy, point-read workloads (e.g., 90% reads, same hot keys).

### When NOT to use DAX
- Write-heavy workloads (DAX doesn't help writes).
- You need strong consistency on reads (DAX serves eventual).
- Working-set is larger than the DAX cluster RAM.


In [ ]:
# Simulated cache-aside pattern (same idea as DAX, just done in-process).
# This is what most Redis/Memcached DynamoDB caches look like in production.

try:
    dynamodb.Table("Products").delete()
    dynamodb.Table("Products").wait_until_not_exists()
except ClientError:
    pass

products = dynamodb.create_table(
    TableName="Products",
    KeySchema=[{"AttributeName": "product_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "product_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
products.wait_until_exists()

with products.batch_writer() as batch:
    for i in range(10):
        batch.put_item(Item={"product_id": f"prod-{i}", "name": f"Widget {i}", "price": Decimal(str(10 + i))})

# --- The cache layer ---
CACHE = {}              # {product_id: item}
db_calls = 0            # counter, for the demo

def get_product_cached(product_id):
    global db_calls
    if product_id in CACHE:
        return CACHE[product_id], "HIT"
    db_calls += 1
    resp = products.get_item(Key={"product_id": product_id})
    item = resp.get("Item")
    if item is not None:
        CACHE[product_id] = item
    return item, "MISS"

# Simulate 1000 page views where 80% of traffic goes to 3 hot products
requests = [f"prod-{random.choice([0,1,2])}" if random.random() < 0.8 else f"prod-{random.randint(0,9)}"
            for _ in range(1000)]

hits = misses = 0
for pid in requests:
    _, status = get_product_cached(pid)
    if status == "HIT": hits += 1
    else: misses += 1

print(f"Simulated 1000 product views:")
print(f"   Cache hits:   {hits}")
print(f"   Cache misses: {misses}  (actual DynamoDB reads)")
print(f"   Hit rate:     {hits/len(requests)*100:.1f}%")
print()
print(f"💡 With DAX, those {misses} misses become DynamoDB reads, and the {hits} hits are served")
print(f"   from microsecond-latency RAM on the DAX cluster. Massive cost + latency win on hot keys.")


## 🎯 Key Takeaways

| Feature | Bad | Best |
|---|---|---|
| **Capacity** | Guessing low RCU/WCU on a spiky launch | On-demand to start; provisioned+auto-scaling once stable |
| **Expiry** | Nightly scan-and-delete cron | TTL on a Unix-epoch attribute (free deletes) |
| **Concurrent updates** | Read → modify → write | Atomic counters, or `ConditionExpression` with a `version` attribute |
| **Idempotency** | Accept duplicate `put_item` | `attribute_not_exists(pk)` condition |
| **Multi-item correctness** | Two independent writes | `TransactWriteItems` (up to 100 items, 2× WCU) |
| **Hot partition** | Partition key = constant / low cardinality | Write sharding: `logical_key#<0..N-1>`, fan-out reads |
| **Hot-key reads** | Millions of identical RCU-burning reads | DAX (AWS) or cache-aside with Redis (µs latency, fewer reads) |

### Rules of thumb

1. **Default to on-demand capacity** until you have real traffic data.
2. **Enable TTL** on anything with a natural expiry (sessions, tokens, caches).
3. **Always use conditional writes** on anything you'd do a read-modify-write for.
4. **Reach for transactions** when invariants span multiple items (transfers, inventory, bookings).
5. **Shard any partition key whose value is fixed / low cardinality** (hot bucket, hot tenant, today's date).
6. **Cache hot reads** — DAX if you're all-in on AWS, Redis/Memcached if you're multi-cloud.

### 🎉 Series Complete

You've now walked through:

1. Partition + sort keys
2. GSI + LSI
3. Single-table design
4. Streams + CDC
5. Capacity, TTL, transactions, hot partitions, DAX

Each of these lines up with things an interviewer will probe ("how do you prevent a lost update?", "how do you fix a hot partition?", "how do you keep Elasticsearch in sync?"), so you can now speak to DynamoDB end-to-end.
